# Models:
**1. BiLSTM with Attention**\
**2. BERT (DistilBERT)**

In [63]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
import random
import warnings
import os

warnings.filterwarnings('ignore')

In [65]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [67]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("ADVANCED MODELS - DEEP LEARNING")

Using device: cpu
ADVANCED MODELS - DEEP LEARNING


# 1. LOAD DATA

In [70]:
print("\nLoading preprocessed datasets")

# Load processed datasets
train_df = pd.read_csv('../data/processed/train.csv')
val_df   = pd.read_csv('../data/processed/val.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

# Load label encoder
with open('../models/label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

# Basic sanity checks
assert not train_df.empty, "Train dataset is empty"
assert not val_df.empty, "Validation dataset is empty"
assert not test_df.empty, " Test dataset is empty"

required_columns = {'text', 'label'}
assert required_columns.issubset(train_df.columns), "Missing columns in train set"
assert required_columns.issubset(val_df.columns), " Missing columns in validation set"
assert required_columns.issubset(test_df.columns), "Missing columns in test set"

print(f" Train samples: {len(train_df)}")
print(f" Val samples:   {len(val_df)}")
print(f" Test samples:  {len(test_df)}")

print(f" Number of classes: {len(label_encoder.classes_)}")
print(f"Class labels: {list(label_encoder.classes_)}")


Loading preprocessed datasets
 Train samples: 35199
 Val samples:   7519
 Test samples:  7539
 Number of classes: 7
Class labels: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']


# 2. BILSTM WITH ATTENTION MODEL

In [75]:
print("\nBuilding BiLSTM with Attention")

# Attention Layer
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_outputs):
        # lstm_outputs: (batch, seq_len, hidden_dim*2)
        scores = self.attn(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights * lstm_outputs, dim=1)
        return context

# BiLSTM + Attention Model
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=dropout
        )

        self.attention = Attention(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)
        context = self.dropout(context)
        return self.fc(context)

# BiLSTM Evaluation Function 
def evaluate_bilstm(model, loader, device):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            outputs = model(x)
            predictions = torch.argmax(outputs, dim=1).cpu().numpy()

            preds.extend(predictions)
            true.extend(y.numpy())

    return np.array(preds), np.array(true)


from collections import Counter

MAX_VOCAB_SIZE = 10000
MAX_LEN = 100

def build_vocab(texts, max_size):
    counter = Counter()
    for text in texts:
        counter.update(text.split())

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)

    return vocab


def encode_text(text, vocab):
    return [vocab.get(word, vocab["<UNK>"]) for word in text.split()]


def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        return seq + [0] * (max_len - len(seq))
    return seq[:max_len]
    
vocab = build_vocab(train_df['text'], MAX_VOCAB_SIZE)

X_train = [pad_sequence(encode_text(t, vocab), MAX_LEN) for t in train_df['text']]
X_val   = [pad_sequence(encode_text(t, vocab), MAX_LEN) for t in val_df['text']]
X_test  = [pad_sequence(encode_text(t, vocab), MAX_LEN) for t in test_df['text']]

X_train = torch.tensor(X_train, dtype=torch.long)
X_val   = torch.tensor(X_val, dtype=torch.long)
X_test  = torch.tensor(X_test, dtype=torch.long)


# Dataset Class
class TextDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.inputs[idx], self.labels[idx]


train_dataset = TextDataset(X_train, train_df['label'].values)
val_dataset   = TextDataset(X_val, val_df['label'].values)
test_dataset  = TextDataset(X_test, test_df['label'].values)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

print(f"Vocabulary size: {len(vocab)}")

# SAVE BiLSTM MODEL 
os.makedirs('../models', exist_ok=True)

torch.save(
    bilstm_model.state_dict(),
    '../models/bilstm_attention.pth'
)

print(" BiLSTM model saved to ../models/bilstm_attention.pth")                    


Building BiLSTM with Attention
Vocabulary size: 10000
 BiLSTM model saved to ../models/bilstm_attention.pth


# 3. RESULTS COMPARISON

In [78]:
print("\nComparing model (Baseline + Deep Learning)")

# Ensure results dict exists
if 'results' not in globals():
    results = {}

# Add BiLSTM results only if metrics exist
if all(var in globals() for var in ['test_acc_bilstm', 'test_p_bilstm', 'test_r_bilstm', 'test_f1_bilstm']):
    results['BiLSTM + Attention'] = {
        'accuracy': test_acc_bilstm,
        'precision': test_p_bilstm,
        'recall': test_r_bilstm,
        'f1': test_f1_bilstm
    }
    print(" BiLSTM results added")
else:
    print(" BiLSTM metrics not found — skipping")

# Build comparison table
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results]
}).sort_values('F1-Score', ascending=False)

print("\n Model Comparison:")
print(comparison_df.to_string(index=False))


Comparing model (Baseline + Deep Learning)
 BiLSTM metrics not found — skipping

 Model Comparison:
     Model  Accuracy  Precision   Recall  F1-Score
DistilBERT  0.789229   0.788907 0.789229  0.788531


# 4. PREPARE BERT DATA

In [81]:
print("\n Preparing BERT data")

from transformers import DistilBertTokenizer

tokenizer_bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset_bert = BERTDataset(
    train_df['text'].values,
    train_df['label'].values,
    tokenizer_bert
)

val_dataset_bert = BERTDataset(
    val_df['text'].values,
    val_df['label'].values,
    tokenizer_bert
)

test_dataset_bert = BERTDataset(
    test_df['text'].values,
    test_df['label'].values,
    tokenizer_bert
)

train_loader_bert = DataLoader(train_dataset_bert, batch_size=16, shuffle=True)
val_loader_bert = DataLoader(val_dataset_bert, batch_size=16)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=16)

print("BERT DataLoaders created")


 Preparing BERT data
BERT DataLoaders created


# 5. TRAIN DISTILBERT MODEL

In [21]:
print("\nTraining DistilBERT")

num_classes = len(label_encoder.classes_)

bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    bert_model.parameters(),
    lr=2e-5,
    correct_bias=False
)

epochs = 5
total_steps = len(train_loader_bert) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

def train_bert_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def eval_bert(model, loader, device):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    return np.array(predictions), np.array(true_labels)


best_f1 = 0
bert_history = {'train_loss': [], 'val_f1': []}

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss = train_bert_epoch(
        bert_model, train_loader_bert, optimizer, scheduler, device
    )

    val_preds, val_true = eval_bert(bert_model, val_loader_bert, device)
    _, _, val_f1, _ = precision_recall_fscore_support(
        val_true, val_preds, average='weighted'
    )

    bert_history['train_loss'].append(train_loss)
    bert_history['val_f1'].append(val_f1)

    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val F1:     {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(bert_model.state_dict(), '../models/distilbert_best.pth')

print(f"\n Best DistilBERT Validation F1: {best_f1:.4f}")


Training DistilBERT


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias', 'pre_classifier.weight', 'pre_classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Epoch 1/5


Evaluating: 100%|█████████████████████████████| 470/470 [03:38<00:00,  2.15it/s]


  Train Loss: 0.7969
  Val F1:     0.7757

Epoch 2/5


...

  Train Loss: 0.3753
  Val F1:     0.7946

Epoch 4/5


Evaluating: 100%|█████████████████████████████| 470/470 [03:43<00:00,  2.10it/s]


  Train Loss: 0.2825
  Val F1:     0.7930

Epoch 5/5


Evaluating: 100%|█████████████████████████████| 470/470 [03:36<00:00,  2.17it/s]

  Train Loss: 0.2164
  Val F1:     0.7890

 Best DistilBERT Validation F1: 0.7946


# 6. DISTILBERT TEST EVALUATION

In [24]:
print("\nEvaluating DistilBERT on Test Set")

bert_model.load_state_dict(torch.load('../models/distilbert_best.pth'))

test_preds_bert, test_true_bert = eval_bert(
    bert_model, test_loader_bert, device
)

test_acc_bert = accuracy_score(test_true_bert, test_preds_bert)
test_p_bert, test_r_bert, test_f1_bert, _ = precision_recall_fscore_support(
    test_true_bert, test_preds_bert, average='weighted'
)

print("\n DistilBERT Test Results:")
print(f"  Accuracy:  {test_acc_bert:.4f}")
print(f"  Precision: {test_p_bert:.4f}")
print(f"  Recall:    {test_r_bert:.4f}")
print(f"  F1-Score:  {test_f1_bert:.4f}")


Evaluating DistilBERT on Test Set


Evaluating: 100%|█████████████████████████████| 472/472 [03:07<00:00,  2.52it/s]


 DistilBERT Test Results:
  Accuracy:  0.7892
  Precision: 0.7889
  Recall:    0.7892
  F1-Score:  0.7885


**ADD DISTILBERT TO RESULTS**

In [28]:
results['DistilBERT'] = {
    'accuracy': test_acc_bert,
    'precision': test_p_bert,
    'recall': test_r_bert,
    'f1': test_f1_bert
}

print(" DistilBERT added to results")

 DistilBERT added to results


# 7. SAVE ADVANCED MODEL RESULTS

**SAFETY CHECK: Ensure Test Metrics Exist**

In [88]:
print("\nChecking test metrics availability")

# BiLSTM SAFETY
if 'test_acc_bilstm' not in globals():
    print(" BiLSTM test metrics missing. Recomputing...")

    bilstm_model = BiLSTMAttention(
        vocab_size=len(vocab),         
        embed_dim=100,                  
        hidden_dim=128,
        num_classes=len(label_encoder.classes_),
        dropout=0.3
    ).to(device)

    # Load trained weights
    bilstm_model.load_state_dict(
        torch.load('../models/bilstm_attention.pth', map_location=device)
    )
    bilstm_model.eval()

    # Recompute test predictions
    test_preds_bilstm, test_true = evaluate_bilstm(  
        bilstm_model, test_loader, device
    )

    # Metrics
    test_acc_bilstm = accuracy_score(test_true, test_preds_bilstm)
    test_p_bilstm, test_r_bilstm, test_f1_bilstm, _ = precision_recall_fscore_support(
        test_true, test_preds_bilstm, average='weighted'
    )

    print("✓ BiLSTM test metrics recomputed")

# BERT SAFETY 
if 'test_acc_bert' not in globals():
    print(" BERT test metrics missing. Recomputing...")

    bert_model = DistilBertForSequenceClassification.from_pretrained(
        '../models/bert_model'
    ).to(device)
    bert_model.eval()

    test_preds_bert, test_true_bert = [], []
    with torch.no_grad():
        for batch in test_loader_bert:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            test_preds_bert.extend(preds)
            test_true_bert.extend(batch['labels'].numpy())

    test_acc_bert = accuracy_score(test_true_bert, test_preds_bert)
    test_p_bert, test_r_bert, test_f1_bert, _ = precision_recall_fscore_support(
        test_true_bert, test_preds_bert, average='weighted'
    )

    print(" BERT test metrics recomputed")

print(" All required test metrics are now available")


Checking test metrics availability
 All required test metrics are now available


In [92]:
print("\n Saving results")

assert 'test_acc_bilstm' in globals(), "BiLSTM test metrics missing"
assert 'test_acc_bert' in globals(), "BERT test metrics missing"

# Collect results
advanced_results = {
    'BiLSTM_Attention': {
        'accuracy': float(test_acc_bilstm),
        'precision': float(test_p_bilstm),
        'recall': float(test_r_bilstm),
        'f1': float(test_f1_bilstm)
    },
    'DistilBERT': {
        'accuracy': float(test_acc_bert),
        'precision': float(test_p_bert),
        'recall': float(test_r_bert),
        'f1': float(test_f1_bert)
    }
}

# Save to JSON

import json
with open('../results/metrics/advanced_results.json', 'w') as f:
    json.dump(advanced_results, f, indent=4)

print(" Saved: results/metrics/advanced_results.json")

print("\nFinal Test F1-Scores:")
print(f"  BiLSTM + Attention : {test_f1_bilstm:.4f}")
print(f"  DistilBERT         : {test_f1_bert:.4f}")


 Saving results
 Saved: results/metrics/advanced_results.json

Final Test F1-Scores:
  BiLSTM + Attention : 0.0047
  DistilBERT         : 0.7885
